# Thử thêm một prompt đối tượng trên hai ảnh VALID
Gắn **csa-prompt-full-karpathy-v175** và **csa-scst-full-karpathy-cache-v1**, bật Internet, chọn GPU T4. Dùng secret có quyền đọc output v94 hoặc điền đường dẫn checkpoint v94 đã mount vào CHECKPOINT_PATH. Gửi ZIP review sau khi chạy.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/94'
SECRET_NAME='codex'
CHECKPOINT_PATH=''  # Optional exact v94 pilot_checkpoint.bin mounted from authorized Output/Dataset
RESUME_SHA='80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24'
WALL_HOURS=1 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_cider_object_slot_valid2')
TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}
CACHE_ROOT='/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1'
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for VALID patches and checkpoint')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)


RECEIPT={'sha256': '80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24', 'updates': 1500, 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'parent_sha256': '56a90b2223b3bb64e433a342d048389dc3e80693fa574c2c1543e31e39a4377b'}
VOCAB_SHA='cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd'
PROMPT_PINS={'prompt_texts.json': '35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2', 'prompt_embeddings.npy': 'f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c'}
CASES={239894: {'coco_id': 239894, 'label': 'teddy bear', 'text': 'a teddy bear', 'roi': {'roi': 1, 'label': 'teddy bear', 'confidence': 0.926, 'bbox': [0.5461971830985916, 0.43859375, 0.793169014084507, 0.704875]}, 'linked': {'coco_id': 239894, 'filename': 'COCO_val2014_000000239894.jpg', 'group': 'val', 'roi_count': 11, 'source_mode': 'object_only', 'objects': [{'instance_id': 'det_0270b3a10178', 'upt_index': 0, 'roi': 0, 'label': 'person', 'bbox': [0.2610907554626465, 0.11613607922759679, 0.7919361877441407, 0.9934927680708784], 'roi_bbox': [0.27180751388621444, 0.12229686975479126, 0.7944600996277142, 0.9990468978881836], 'IoU': 0.9623897040803431, 'confidence': 0.995604395866394, 'text': 'a person'}], 'relations': [], 'counts': {'objects': 1, 'relations': 0, 'mapped_objects': 1, 'kept_objects': 1, 'mapped_relations': 0, 'capped_objects': 0}}, 'historic_beams': ['a woman in a black outfit holding a red bag', 'a woman in a black dress holding a red bag', 'a woman in a black outfit holding a red object', 'a woman in a black dress holding a red object', 'a woman in a black outfit holding a red bag and a red bag'], 'historic_rerank': 'a woman in a black outfit holding a red bag'}, 487438: {'coco_id': 487438, 'label': 'bed', 'text': 'a bed', 'roi': {'roi': 1, 'label': 'bed', 'confidence': 0.8526, 'bbox': [0.160296875, 0.48352083333333334, 0.7308125000000001, 0.9886666666666667]}, 'linked': {'coco_id': 487438, 'filename': 'COCO_val2014_000000487438.jpg', 'group': 'val', 'roi_count': 14, 'source_mode': 'object_only', 'objects': [{'instance_id': 'det_ae24eaa745e8', 'upt_index': 1, 'roi': 0, 'label': 'suitcase', 'bbox': [0.19244177211740005, 0.80328369140625, 0.5194768565680699, 0.9949711608886719], 'roi_bbox': [0.19817187786102294, 0.8068958282470703, 0.5137499809265137, 0.9912708282470704], 'IoU': 0.9281556741271637, 'confidence': 0.9869580864906311, 'text': 'a suitcase'}, {'instance_id': 'det_261e0a8e7ee9', 'upt_index': 2, 'roi': 6, 'label': 'backpack', 'bbox': [0.37074377120771285, 0.5315413665771485, 0.557257217493111, 0.7438865661621094], 'roi_bbox': [0.3778125047683716, 0.5420208613077799, 0.5994531154632569, 0.7419375101725261], 'IoU': 0.7467384560899055, 'confidence': 0.8067554235458374, 'text': 'a backpack'}], 'relations': [], 'counts': {'objects': 3, 'relations': 0, 'unmatched_objects': 1, 'mapped_objects': 2, 'kept_objects': 2, 'mapped_relations': 0, 'capped_objects': 0}}, 'historic_beams': ['a green chair sitting in a green room', 'a room filled with a chair and a suitcase', 'a room filled with a chair and a chair', 'a room with a desk and a chair', 'a room with a chair and a suitcase'], 'historic_rerank': 'a room filled with a chair and a suitcase'}}


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Probe parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Probe active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Probe parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    links,=PROMPT_TEST_ROOT.rglob('prompt_links.jsonl')
    raw_texts=(links.parent/'prompt_texts.json').read_bytes()
    if hashlib.sha256(raw_texts).hexdigest()!=PROMPT_PINS['prompt_texts.json']:
        raise ValueError('Wrong v175 text list')
    texts=json.loads(raw_texts)
    missing={case['text'] for case in CASES.values()}-set(texts)
    if missing:
        raise ValueError('Missing v175 cached phrases: '+', '.join(sorted(missing)))
    if CHECKPOINT_PATH:
        weights=Path(CHECKPOINT_PATH)
        if not weights.is_file():
            raise FileNotFoundError(f'Mounted checkpoint not found: {weights}')
    else:
        from kaggle_secrets import UserSecretsClient
        try:
            token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        except Exception:
            raise RuntimeError('Enable the named Kaggle API Secret with access to the source notebook') from None
        if not token:
            raise RuntimeError('The named Kaggle API Secret is empty')
        TOKENS.append(token)
        cli_env=os.environ.copy()
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
            cli_env.pop(key,None)
        run([sys.executable,'-m','pip','install','-q','--target',CLI,'kaggle==2.2.5'],300,cli_env)
        cli_env.update(KAGGLE_API_TOKEN=token,PYTHONPATH=str(CLI)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1')
        downloaded=SOURCE/'scst_v94'
        downloaded.mkdir()
        run([sys.executable,'-u','-c','from kaggle.cli import main; main()',
             'kernels','output',SOURCE_VERSION,'-p',downloaded,
             '--file-pattern',r'(^|/)update1500/pilot_checkpoint[.]bin$'],900,cli_env)
        candidates=list(downloaded.rglob('pilot_checkpoint.bin'))
        if len(candidates)!=1:
            raise ValueError('Expected the SCST1500 checkpoint from exact v94')
        weights=candidates[0]
    with weights.open('rb') as stream:
        if hashlib.file_digest(stream,'sha256').hexdigest()!=RESUME_SHA:
            raise ValueError('Changed SCST1500 checkpoint')

    sys.path.insert(0,str(SOURCE))
    (SOURCE/'captioning').mkdir(exist_ok=True)
    (SOURCE/'captioning/__init__.py').touch()
    for path,name in MAPPING.items():
        if name in {'semantic_data.py','patch_model.py','patch_beam.py','patch_data.py','full_data.py','data.py','real_bank_data.py','cuda_runtime_guard.py'}:
            download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    for path,digest in TRAIN_PINS.items():
        name='bank_layers.py' if path.endswith('/model.py') else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+TRAIN_REVISION+'/'+path,digest,SOURCE/name)
    import numpy as np
    import torch
    from csa_patch_model import CSAPatchCaptioner
    from patch_beam import beam_search, check_paths, text_from_ids
    from semantic_data import record_arrays
    from real_bank_data import sha

    if torch.__version__ not in ('2.11.0+cu128','2.11.0+cu130') or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU T4 with CUDA-enabled Torch 2.11.0')
    device=torch.device('cuda:0')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False

    mount=Path(CACHE_ROOT)
    with zipfile.ZipFile(mount/'cache_metadata_zip.bin') as z:
        receipt_bytes=z.read('part0/scst_cache_receipt.json')
        if hashlib.sha256(receipt_bytes).hexdigest()!='3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510':
            raise ValueError('Wrong SCST Part 0 receipt')
        cache=json.loads(receipt_bytes)['files_sha256']
        def member(name):
            raw=z.read('part0/'+name)
            if hashlib.sha256(raw).hexdigest()!=cache[name]:
                raise ValueError('Changed cache member '+name)
            return raw
        import io
        valid_ids=np.load(io.BytesIO(member('valid/ids.npy')),allow_pickle=False)
        raw_vocab=member('vocabulary.json')
        if hashlib.sha256(raw_vocab).hexdigest()!=VOCAB_SHA:
            raise ValueError('Wrong vocabulary for SCST1500')
        words=json.loads(raw_vocab)
    patchfile=mount/'part0_valid_patches.npy'
    if sha(patchfile)!=cache['valid/patches.npy']:
        raise ValueError('Changed VALID patches')
    patches=np.load(patchfile,mmap_mode='r',allow_pickle=False)
    index={int(cid):i for i,cid in enumerate(valid_ids)}
    if len(index)!=5000 or not CASES.keys()<=index.keys():
        raise ValueError('Wrong VALID coverage')
    if sha(links)!='e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0':
        raise ValueError('Wrong prompt175 links')
    vector_path=links.parent/'prompt_embeddings.npy'
    for path,digest in PROMPT_PINS.items():
        if sha(links.parent/path)!=digest:
            raise ValueError('Wrong prompt175 asset '+path)
    vectors=np.load(vector_path,allow_pickle=False)
    with links.open(encoding='utf8') as stream:
        original={r['coco_id']:r for r in map(json.loads,stream) if r['group']=='val' and r['coco_id'] in CASES}
    if set(original)!=set(CASES):
        raise ValueError('Missing original VAL prompts')
    from copy import deepcopy
    import re

    def augmented(row, case):
        roi = case['roi']
        if len(row['objects']) >= 8 or any(obj['roi'] == roi['roi'] for obj in row['objects']):
            raise ValueError('Expected one unused detector region and one empty object slot')
        edited = deepcopy(row)
        edited['objects'].append(dict(instance_id='verified_yolo_' + str(case['coco_id']),
                                     roi=roi['roi'], text=case['text'],
                                     bbox=roi['bbox'], roi_bbox=roi['bbox']))
        return edited

    def mentions(caption, label):
        suffix = r's?'
        return bool(re.search(r'\b' + re.escape(label) + suffix + r'\b', caption.lower()))

    def verify_slot_change(before, after, slot):
        modified = {'object_text_id','object_box','object_weights','object_valid'}
        for name in before:
            if name in modified:
                np.testing.assert_array_equal(np.delete(before[name],slot,axis=0),
                                              np.delete(after[name],slot,axis=0))
            else:
                np.testing.assert_array_equal(before[name],after[name])
        if before['object_valid'][slot] or not after['object_valid'][slot]:
            raise ValueError('Expected exactly one newly active object slot')


    ck=torch.load(weights,map_location='cpu',weights_only=False)
    for field in ('updates','source_sha256','parent_sha256'):
        if ck[field]!=RECEIPT[field]:
            raise ValueError('Wrong checkpoint lineage')
    model=CSAPatchCaptioner(len(words),ck['model']['core.nouns'],ck['model']['core.verbs'])
    model.load_state_dict(ck['model'],strict=True)
    del ck
    model.to(device).eval()

    def encode(row,x):
        arrays=record_arrays(row,texts)
        prompt={k:v for k,v in arrays.items() if k not in ('object_text_id','relation_text_id')}
        for role in ('object','relation'):
            prompt[role+'_text']=vectors[arrays[role+'_text_id']]*arrays[role+'_valid'][:,None]
        prompt={k:torch.from_numpy(np.ascontiguousarray(v[None])).to(device) for k,v in prompt.items()}
        return model.encode(x,prompt),arrays

    def measure(memory, label):
        seq,scores=beam_search(model,memory,width=5,maximum=20)
        check_paths(model,memory,seq,scores)
        captions=[text_from_ids(ids.tolist(),words) for ids in seq[0]]
        if len(captions)!=5:
            raise ValueError('Expected five beams')
        return dict(captions=captions,
                    target_ranks=[i for i,c in enumerate(captions,1) if mentions(c,label)],
                    scores=scores[0].tolist())

    results=[]
    with torch.inference_mode():
        for cid,row in original.items():
            if time.monotonic()>=PARENT_DEADLINE:
                raise TimeoutError('VALID inference deadline')
            case=CASES[cid]
            if row!=case['linked']:
                raise ValueError('Changed audited links')
            x=torch.from_numpy(np.array(patches[index[cid]:index[cid]+1],dtype=np.float32)).to(device)
            first,before=encode(row,x)
            second,after=encode(augmented(row,case),x)
            verify_slot_change(before,after,len(row['objects']))
            baseline=measure(first,case['label'])
            intervention=measure(second,case['label'])
            results.append(dict(coco_id=cid,prompt=case['text'],roi=case['roi'],
                                archived_rerank=case['historic_rerank'],
                                archived_beams=case['historic_beams'],
                                archived_beams_match=baseline['captions']==case['historic_beams'],
                                original=baseline,added_slot=intervention))
    if len(results)!=len(CASES):
        raise ValueError('Incomplete paired inference')
    result=dict(split='val',images=len(results),checkpoint_sha256=RESUME_SHA,records=results)
    (OUTPUT/'result.json').write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')
    complete=True
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('result.json','notebook.log'):
            path=OUTPUT/name
            if path.is_file():z.write(path,name)
    print('SEND REVIEW:',review, 'complete:',complete,flush=True)
